# Feature Engineering — Job Scam Analyzer

**Three types of features **
1. **Structural flags** — is salary missing? is company profile empty? (strongest scam signals)
2. **Red flag lexicon counts** — urgency words, upfront payment language, vague promises
3. **TF-IDF text features** — top 300 keywords from job posting text



In [3]:
import pandas as pd
import numpy as np
import sqlite3
import re
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import scipy.sparse as sp

print(" Imports done")

 Imports done


In [4]:
# Reload and re-clean (same as analysis.py block 2)
df = pd.read_csv("../data/raw/fake_job_postings.csv")

text_cols = ['company_profile','description','requirements',
             'benefits','title','location','department',
             'salary_range','employment_type',
             'required_experience','required_education',
             'industry','function']

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna('').astype(str).str.strip()

df['has_company_profile'] = (df['company_profile'] != '').astype(int)
df['has_salary']          = (df['salary_range']    != '').astype(int)
df['has_requirements']    = (df['requirements']    != '').astype(int)
df['has_benefits']        = (df['benefits']        != '').astype(int)
df['has_logo']            = df['has_company_logo'].fillna(0).astype(int)
df['telecommuting']       = df['telecommuting'].fillna(0).astype(int)

df['full_text'] = (df['title']           + ' ' +
                   df['company_profile'] + ' ' +
                   df['description']     + ' ' +
                   df['requirements'])

print(f"  Data loaded: {df.shape}")
print(f"  Fraud rate: {df['fraudulent'].mean()*100:.1f}%")

  Data loaded: (17880, 24)
  Fraud rate: 4.8%


## Feature Group 1 — Red Flag Lexicon

defining word lists for 4 scam patterns and count how many times
each pattern appears in the full posting text.

These are the most *human-interpretable* features — they directly
map to what a fraud analyst would look for manually.

In [5]:
URGENCY_WORDS = [
    'urgent','immediate','asap','hurry','limited slots',
    'act now','fast','quick','rush','apply immediately'
]
MONEY_UPFRONT = [
    'pay upfront','registration fee','training fee',
    'deposit required','starter kit','fee required',
    'investment required','pay to join'
]
VAGUE_PROMISES = [
    'work from home','be your own boss','financial freedom',
    'unlimited earning','passive income','multi-level',
    'mlm','network marketing','make money fast'
]
NO_CREDENTIAL = [
    'no experience required','no qualification',
    'anyone can apply','guaranteed income',
    'guaranteed job','no degree'
]

def flag_count(text, wordlist):
    text = text.lower()
    return sum(1 for w in wordlist if w in text)

df['urgency_count']    = df['full_text'].apply(lambda x: flag_count(x, URGENCY_WORDS))
df['money_flag_count'] = df['full_text'].apply(lambda x: flag_count(x, MONEY_UPFRONT))
df['vague_count']      = df['full_text'].apply(lambda x: flag_count(x, VAGUE_PROMISES))
df['credential_count'] = df['full_text'].apply(lambda x: flag_count(x, NO_CREDENTIAL))

# Quick check — do scam posts score higher on these?
print("Average flag counts: Scam vs Legit\n")
flag_cols = ['urgency_count','money_flag_count','vague_count','credential_count']
print(df.groupby('fraudulent')[flag_cols].mean().round(3))

Average flag counts: Scam vs Legit

            urgency_count  money_flag_count  vague_count  credential_count
fraudulent                                                                
0                   0.520             0.000        0.008             0.002
1                   0.408             0.001        0.130             0.017


## Feature Group 2 — Text Length Features

Scam postings tend to be short 

In [6]:
df['desc_len']        = df['description'].str.len()
df['profile_len']     = df['company_profile'].str.len()
df['req_len']         = df['requirements'].str.len()
df['title_word_cnt']  = df['title'].str.split().str.len()

# Compare lengths: scam vs legit
print("Average text lengths: Scam vs Legit\n")
len_cols = ['desc_len','profile_len','req_len','title_word_cnt']
print(df.groupby('fraudulent')[len_cols].mean().round(1))

Average text lengths: Scam vs Legit

            desc_len  profile_len  req_len  title_word_cnt
fraudulent                                                
0             1220.0        640.6    597.3             3.7
1             1154.3        228.4    445.7             4.0


## Feature Group 3 — Categorical Encoding

Employment type, required experience, and education level
are label-encoded so the model can use them numerically.

In [7]:
le = LabelEncoder()

df['employment_type_enc'] = le.fit_transform(
    df['employment_type'].fillna('Unknown'))

df['required_exp_enc'] = le.fit_transform(
    df['required_experience'].fillna('Unknown'))

df['required_edu_enc'] = le.fit_transform(
    df['required_education'].fillna('Unknown'))

print(" Categoricals encoded")
print(df[['employment_type','employment_type_enc']].drop_duplicates().head(8))

 Categoricals encoded
   employment_type  employment_type_enc
0            Other                    3
1        Full-time                    2
2                                     0
9        Part-time                    4
35        Contract                    1
82       Temporary                    5


## Feature Group 4 — TF-IDF Text Features

TF-IDF converts raw posting text into 300 numerical keyword
features. Bigrams (2-word phrases) are included because
scam phrases like "work from home" or "no experience" are
more meaningful as phrases than single words.

In [8]:
tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),       # unigrams + bigrams
    min_df=5,                 # must appear in at least 5 posts
    stop_words='english',
    strip_accents='unicode'
)

tfidf_matrix = tfidf.fit_transform(df['full_text'])

print(f"  TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"  Sample features: {tfidf.get_feature_names_out()[:10].tolist()}")

  TF-IDF matrix shape: (17880, 300)
  Sample features: ['ability', 'able', 'account', 'activities', 'agency', 'amp', 'analysis', 'analytics', 'application', 'applications']


## Assemble Final Feature Matrix

Combine all 4 feature groups into one matrix X.
Final shape will be: (17880, 317)
— 17 structured features + 300 TF-IDF features

In [9]:
STRUCTURED_FEATURES = [
    # Missing field flags
    'has_company_profile','has_salary','has_requirements',
    'has_benefits','has_logo','telecommuting',
    # Red flag counts
    'urgency_count','money_flag_count','vague_count','credential_count',
    # Text lengths
    'desc_len','profile_len','req_len','title_word_cnt',
    # Encoded categoricals
    'employment_type_enc','required_exp_enc','required_edu_enc',
]

X_structured = df[STRUCTURED_FEATURES].fillna(0).values
X_tfidf      = tfidf_matrix.toarray()

# Stack horizontally
X = np.hstack([X_structured, X_tfidf])
y = df['fraudulent'].values

feature_names = STRUCTURED_FEATURES + tfidf.get_feature_names_out().tolist()

print(f"  Final feature matrix: {X.shape}")
print(f"  Total features: {len(feature_names)}")
print(f"  Target: {y.sum()} fraud out of {len(y)} total")

  Final feature matrix: (17880, 317)
  Total features: 317
  Target: 866 fraud out of 17880 total


In [10]:
# Save feature matrix
np.savez('./data/features.npz', X=X, y=y)

# Save feature names
np.save('./data/feature_names.npy', np.array(feature_names, dtype=object))

# Save tfidf vectorizer for use in Streamlit app later
import joblib
joblib.dump(tfidf, './outputs/tfidf.pkl')

# Save df with all engineered columns for Power BI export later
df.to_csv('./data/df_engineered.csv', index=False)

print("    Saved:")
print("    ./data/features.npz       ← X and y arrays")
print("    ./data/feature_names.npy  ← feature name list")
print("    ./outputs/tfidf.pkl       ← TF-IDF vectorizer")
print("    ./data/df_engineered.csv  ← full dataframe with all features")

    Saved:
    ./data/features.npz       ← X and y arrays
    ./data/feature_names.npy  ← feature name list
    ./outputs/tfidf.pkl       ← TF-IDF vectorizer
    ./data/df_engineered.csv  ← full dataframe with all features
